# Module 1: Math Foundations (Zero to Hero)

If you haven't touched math in years, don't worry! This notebook is designed to build your skills from the ground up.
Transformers speak the language of **Linear Algebra**, but at its core, it's just about organizing and comparing numbers.

## 1. What is a Vector and a Matrix?

### The "Fruit Feature" Analogy
Imagine you are describing a **Fruit** to a computer.

| Feature | Value |
| :--- | :--- |
| **Sweetness** | 0.9 |
| **Crunchiness**| 0.8 |
| **Color (Red)**| 0.2 |

- **Vector**: A single list of numbers (a word). `[0.9, 0.8, 0.2]` is the word's "numerical identity".
- **Matrix**: A grid of vectors (a sentence). A 3-word sentence is a 3-row matrix.
- **Tensor**: A stack of matrices (a batch of sentences).

**The Output**: Numbers that the computer can use for geometry and similarity.

In [ ]:
import torch

# Reproducibility: fix the random seed so every run gives the same numbers.
torch.manual_seed(0)

# A Vector (3 features)
word_vector = torch.tensor([0.9, 0.8, 0.2])

# A Matrix (2 words, 3 features)
sentence_matrix = torch.tensor([
    [0.9, 0.8, 0.2], # Word 1
    [0.1, 0.2, 0.8]  # Word 2
])

print(f"Vector shape: {word_vector.shape}")
print(f"Matrix shape (Words, Features): {sentence_matrix.shape}")

## 2. Transpose (Alignment)

### The Concept
Transposing flips a matrix. Rows become columns. Written as $\mathbf{A}^T$.

### Why?
A dot product requires vectors to align. We transpose the **Key** matrix in attention so its features "dock" perfectly with the **Query**'s features.

In [ ]:
A = torch.tensor([[1, 2, 3], [4, 5, 6]])
print("Original (2x3):\n", A)
print("Transposed (3x2):\n", A.T)

## 3. Dot Product (Similarity Score)

### The Math
$$\mathbf{a} \cdot \mathbf{b} = \sum_{i=1}^{n} a_i b_i$$

### Why?
It measures **how much two words overlap**. 
- **High Score**: Similar meaning (e.g., 'King' and 'Royal').
- **Zero Score**: Unrelated (e.g., 'King' and 'Banana').
- **Negative Score**: Opposite (rare in basic attention, but possible).

In [ ]:
v1 = torch.tensor([1.0, 1.0])
v2 = torch.tensor([1.0, 1.0])   # Same direction  -> positive (aligned)
v3 = torch.tensor([-1.0, 1.0])  # Perpendicular   -> zero (unrelated)
v4 = torch.tensor([-1.0, -1.0]) # Opposite        -> negative (anti-aligned)

print(f"Agreement Score (same direction): {torch.dot(v1, v2)}")   # 1*1 + 1*1 = 2
print(f"Zero Score (perpendicular):       {torch.dot(v1, v3)}")   # 1*-1 + 1*1 = 0
print(f"Negative Score (opposite):        {torch.dot(v1, v4)}")   # 1*-1 + 1*-1 = -2

## 4. Matrix Multiplication (Parallel Efficiency)

### Tracking Shapes (CRITICAL 🔥)
In LLMs, we process hundreds of words at once. 
- $Q$: (2 words, 3 features)
- $K^T$: (3 features, 2 words)
- **Result**: (2, 2) grid of similarities.

**The Output**: A matrix where every entry $(i, j)$ is the similarity of word $i$ with word $j$.

In [ ]:
Q = torch.randn(2, 3)
K_T = torch.randn(3, 2)
scores = torch.matmul(Q, K_T)
print(f"Similarity Matrix Shape: {scores.shape}")

## 5. Softmax (Focusing the Model)

### Math
$$\sigma(\mathbf{z})_i = \frac{e^{z_i}}{\sum e^{z_j}}$$

### Why?
We need percentages that sum to 100%. We use $e^x$ because it **exaggerates** differences—making the model pick a clear winner (Focus).

In [ ]:
raw = torch.tensor([10.0, 1.0, 0.1])
probs = torch.softmax(raw, dim=0)
print(f"Attention Focus: {probs}")
print(f"Sums to: {probs.sum():.4f}  (softmax outputs always sum to 1)")

## 6. Numerical Stability (The √dk trick)

### What is $d_k$?
$d_k$ is the **dimension of the Query/Key vectors** — i.e. how many features each Query and Key has. When we compute a score by dotting a Query with a Key, we sum $d_k$ products together. The bigger $d_k$ is, the larger (in magnitude) those scores tend to grow, just from adding up more terms.

### Why big scores hurt learning (Saturation)
Here is the part that is easy to get backwards. If the raw scores (logits) feeding softmax get very large and spread apart, softmax does **not** go "flat" — it does the opposite. It **saturates**: one probability rushes toward ≈1 and the rest collapse toward ≈0 (a "peaky", winner-take-all distribution). The problem is that softmax's gradient is near **zero** exactly in this saturated regime, so the model can barely learn from it — small changes to the scores stop moving the probabilities. (Backprop and gradients are covered in detail in **Notebook 02**.)

### Scaling
To keep scores in a sane range *before* softmax, we divide them by $\sqrt{d_k}$. This shrinks the magnitude so softmax stays responsive (not yet saturated) and gradients keep flowing. The little demo below shows how widening the gap between logits drives softmax toward a saturated, near one-hot output.

In [ ]:
# --- Demo A: big, spread-out logits SATURATE softmax (peaky, near one-hot) ---
big   = torch.tensor([10.0, 1.0, 0.1])   # large gap between the top score and the rest
small = torch.tensor([1.0, 0.1, 0.01])   # same ordering, but scores are close together

print("softmax(big logits)  :", torch.softmax(big, dim=0))
print("  -> one value ~1, others ~0  (SATURATED: gradient here is ~0, hard to learn)\n")

print("softmax(small logits):", torch.softmax(small, dim=0))
print("  -> probabilities stay closer together (responsive: gradients still flow)\n")

# --- Demo B: dividing by sqrt(d_k) tames the variance of the scores ---
torch.manual_seed(0)
d_k = 64
Q = torch.randn(1000, d_k)   # 1000 random Query vectors
K = torch.randn(1000, d_k)   # 1000 random Key vectors
raw_scores    = (Q * K).sum(dim=1)            # dot products WITHOUT scaling
scaled_scores = raw_scores / (d_k ** 0.5)     # dot products WITH /sqrt(d_k)

print(f"d_k = {d_k}")
print(f"Variance of scores BEFORE scaling: {raw_scores.var():.2f}  (grows with d_k)")
print(f"Variance of scores AFTER  /sqrt(d_k): {scaled_scores.var():.2f}  (~1, well-behaved)")

## 7. The Final Boss: Scaled Dot-Product Attention

$$ Attention(Q, K, V) = softmax\left(\frac{QK^T}{\sqrt{d_k}}\right)V $$

### Library Analogy (QKV)
- **Query ($Q$)**: The book title you are searching for.
- **Key ($K$)**: The label on the spine of the book.
- **Value ($V$)**: The actual knowledge inside the book.

**The Output**: A new vector that is a **Weighted Average** (Expectation) of the values based on how well the query matched the keys.

In [ ]:
import torch.nn.functional as F
from math import sqrt

def scaled_dot_product_attention(query, key, value, mask=None):
    dk = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / sqrt(dk)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    attention_weights = F.softmax(scores, dim=-1)
    output = torch.matmul(attention_weights, value)
    return output, attention_weights

torch.manual_seed(0)
x = torch.randn(1, 3, 4)  # 1 batch, 3 words, 4 features each

# Passing the SAME x as Q, K, and V is what makes this *self*-attention:
# every word builds its Query, Key, and Value from itself and attends over the same sequence.
output, weights = scaled_dot_product_attention(x, x, x)

print("Attention weights (rows = each word's focus over all 3 words):")
print(weights.squeeze(0))
print("\nEach row sums to 1:", weights.squeeze(0).sum(dim=-1))
print("\nFinal Output Layer Shape:", output.shape)

### 🏋️ Try it yourself

1. **Word similarity by hand.** Create three 4-dimensional word vectors of your own (e.g. `king`, `queen`, `banana`). Use `torch.dot` to compute all three pairwise dot products and check whether the two "royal" words score higher with each other than either does with `banana`.
2. **Watch softmax saturate.** Take the logits `[2.0, 1.0, 0.5]`, run softmax, then *multiply the same logits by 10* and run softmax again. Print both results and describe in a comment how the distribution changes (does it get peakier or flatter?).

In [ ]:
import torch

# --- Exercise 1: word similarity ---
king   = torch.tensor([0.9, 0.8, 0.1, 0.7])
queen  = torch.tensor([0.8, 0.9, 0.1, 0.6])
banana = torch.tensor([0.1, 0.0, 0.9, 0.2])

# TODO: print torch.dot(king, queen), torch.dot(king, banana), torch.dot(queen, banana)
# Is king-queen the highest?


# --- Exercise 2: softmax saturation ---
logits = torch.tensor([2.0, 1.0, 0.5])
# TODO: print torch.softmax(logits, dim=0)
# TODO: print torch.softmax(logits * 10, dim=0)
# Comment: peakier or flatter?